# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a walkthrough for loading and exploring the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj) using the `mlcroissant` library. The dataset captures survey-based demographic and psychological metrics, including GAD-7, PHQ-9, and PSQ scores.

### Dataset Source
The dataset is described by a Croissant schema hosted at:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`


In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load dataset
dataset = mlc.Dataset(croissant_url)
# The metadata here is an mlcroissant.Metadata object, which has attribute access
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview
Review available `RecordSet`s, their fields, and their Croissant `@id` references.

In [ ]:
print("Record sets available in the dataset:")
record_sets = dataset.record_sets
for rset in record_sets:
    print(f"- RecordSet @id: {rset.id}")
    print(f"  Name: {rset.name}")
    print(f"  Description: {rset.description}")
    print("  Fields (by @id):")
    for field in rset.fields:
        print(f"      - Field @id: {field.id} | name: {field.name} | data_type: {field.data_type}")
    print("")

## 3. Data Extraction
Load data from each RecordSet using their `@id`. You may replace or augment this cell if additional RecordSets are present in the schema.

In [ ]:
# Gather all record set @id's
record_set_ids = [rs.id for rs in dataset.record_sets]
print(f"RecordSet @ids: {record_set_ids}")

# Load each record set into a pandas DataFrame
dataframes = {}
for recset_id in record_set_ids:
    records = list(dataset.records(record_set=recset_id))
    df = pd.DataFrame(records)
    dataframes[recset_id] = df
    print(f"Loaded @{recset_id} with {len(df)} records and columns: {df.columns.tolist()}")
    print(df.head(2), "\n")

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps such as filtering records, normalization, and grouping. We'll pick numeric and categorical fields by their `@id` for demonstration.

We'll assume the main survey responses are available under a primary RecordSet (e.g., `cr:responses` or similar—**replace with the actual @id as appropriate!**)


In [ ]:
# Find a RecordSet with survey data. You may need to update the record_set_id if it's different.
record_set_id = record_set_ids[0]  # If only one, use it; otherwise pick the main responses RecordSet
df = dataframes[record_set_id]

# For demonstration, pick a known or typical field, e.g., GAD-7 total score
# Find the field by checking available columns & their mapping
print("Available columns in selected RecordSet:")
print(df.columns.tolist())

# For this dataset, suppose 'cr:field_gad7_total' and 'cr:field_age' are present.
# You should check and adapt the field names to actual @ids from the column list above.
# Field IDs (examples, replace with real @ids as printed if different)
gad7_field = None
age_field = None
gender_field = None
for col in df.columns:
    if 'gad7' in col.lower():
        gad7_field = col
    if 'age' in col.lower():
        age_field = col
    if 'gender' in col.lower():
        gender_field = col
# If not found, print warnings:
if not gad7_field:
    print("Warning: GAD-7 field not found, using first numeric column as a placeholder.")
    # pick first numeric field
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            gad7_field = col
            break
if not age_field:
    print("Warning: Age field not found, using second column as placeholder.")
    age_field = df.columns[1]
if not gender_field:
    print("Warning: Gender field not found, using third column as placeholder.")
    gender_field = df.columns[2]

# Filtering records where GAD-7 total > 10
threshold = 10
if gad7_field in df.columns:
    filtered_df = df[df[gad7_field] > threshold].copy()
    print(f"Filtered records with {gad7_field} > {threshold}:")
    print(filtered_df[[gad7_field, age_field, gender_field]].head())

    # Normalize GAD-7 total
    norm_col = f"{gad7_field}_normalized"
    filtered_df[norm_col] = (filtered_df[gad7_field] - filtered_df[gad7_field].mean()) / filtered_df[gad7_field].std()
    print(f"Normalized {gad7_field} for filtered records:")
    print(filtered_df[[gad7_field, norm_col]].head())

    # Group by gender
    group_field = gender_field
    if group_field in df.columns:
        grouped_df = filtered_df.groupby(group_field)[gad7_field].agg(['mean', 'count', 'std'])
        print(f"Grouped statistics of {gad7_field} by {group_field}:")
        print(grouped_df)
else:
    print(f"Field {gad7_field} not found in DataFrame.")

## 5. Visualization
Visualize field distributions and group comparisons with matplotlib and seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,4))
if gad7_field and gad7_field in df.columns:
    sns.histplot(df[gad7_field], bins=15, kde=True, color='skyblue')
    plt.title(f"Distribution of {gad7_field}")
    plt.xlabel(gad7_field)
    plt.ylabel("Count")
    plt.show()

if gender_field in df.columns and gad7_field in df.columns:
    plt.figure(figsize=(8,4))
    sns.boxplot(data=df, x=gender_field, y=gad7_field)
    plt.title(f"{gad7_field} by {gender_field}")
    plt.show()

## 6. Conclusion
In this notebook, we explored the FAIR² mental health survey dataset from Kilifi County using the `mlcroissant` library:

- The dataset includes demographic and mental health indicator variables for survey participants.
- We programmatically loaded all record sets and illustrated field access by `@id`.
- Standard EDA was performed by filtering, normalizing, and aggregating a numeric field (e.g., GAD-7 total score), grouping by a categorical variable (gender), and plotting data distributions.

This approach can be adapted to other datasets and tasks by referencing the specific `@id` values of interest for record sets and fields, ensuring reproducibility and schema-aligned analysis with mlcroissant.